In [1]:
import pandas as pd
import geopandas as gpd
from shared_utils import geo_utils
from calitp_data_analysis.sql import get_engine
db_engine = get_engine()
import gcsfs
import google.auth
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()
from shapely import wkt
from shapely.geometry import LineString
import os

pd.set_option('display.max_columns', None)

In [2]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses'

In [3]:
# Querying GTFS stops for one weekday
with db_engine.connect() as connection:
    query = """
        SELECT *
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_shapes
        WHERE service_date = DATE('2025-03-12')
    """
    shapes_unique = pd.read_sql(query, connection)

/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


In [4]:
shapes_unique.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11217 entries, 0 to 11216
Data columns (total 11 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   key                                          11217 non-null  object        
 1   feed_key                                     11217 non-null  object        
 2   service_date                                 11217 non-null  object        
 3   shape_id                                     11217 non-null  object        
 4   shape_array_key                              11217 non-null  object        
 5   feed_timezone                                11217 non-null  object        
 6   n_trips                                      11217 non-null  int64         
 7   shape_first_departure_datetime_pacific       11217 non-null  datetime64[ns]
 8   shape_last_arrival_datetime_pacific          11217 non-null  datetime64[ns]


In [5]:
def make_linestring(pts):
    if not pts or len(pts) < 2:
        return None
    return LineString([wkt.loads(pt) for pt in pts])

shapes_unique["geometry"] = shapes_unique["pt_array"].apply(make_linestring)

shapes_unique = gpd.GeoDataFrame(
    shapes_unique,
    geometry="geometry",
    crs="EPSG:4326"
)

print(shapes_unique.geometry.notna().sum())
print(shapes_unique.geometry.geom_type.value_counts())

11217
LineString    11217
Name: count, dtype: int64


In [6]:
# with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/shapes_array_cleaned.parquet", "rb") as f:
#     shapes_array_clean = gpd.read_parquet(f)

In [7]:
# shapes_unique.explore()

In [8]:
# shapes_array_clean.info()

In [9]:
with db_engine.connect() as connection:
    query = """
        SELECT
            key,
            trip_id,
            direction_id,
            route_id,
            feed_key,
            shape_array_key,
            shape_id,
            service_date
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-03-12')
    """

    route_id_shapes = pd.read_sql(query, connection)

In [10]:
shapes_unique.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 11217 entries, 0 to 11216
Data columns (total 12 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   key                                          11217 non-null  object        
 1   feed_key                                     11217 non-null  object        
 2   service_date                                 11217 non-null  object        
 3   shape_id                                     11217 non-null  object        
 4   shape_array_key                              11217 non-null  object        
 5   feed_timezone                                11217 non-null  object        
 6   n_trips                                      11217 non-null  int64         
 7   shape_first_departure_datetime_pacific       11217 non-null  datetime64[ns]
 8   shape_last_arrival_datetime_pacific          11217 non-null  datetim

In [11]:
# Number of unique feed + shape combinations
print(
    route_id_shapes[["feed_key", "shape_id"]]
    .drop_duplicates()
    .shape
)

print(
    shapes_unique[["feed_key", "shape_id"]]
    .drop_duplicates()
    .shape
)

(11221, 2)
(11217, 2)


In [12]:
route_pairs = route_id_shapes[["feed_key", "shape_id"]].drop_duplicates()

shape_pairs = shapes_unique[["feed_key", "shape_id"]].drop_duplicates()

missing_pairs = route_pairs.merge(
    shape_pairs,
    on=["feed_key", "shape_id"],
    how="left",
    indicator=True
)

missing_pairs = missing_pairs[
    missing_pairs["_merge"] == "left_only"
].drop(columns="_merge")

print(missing_pairs)

                              feed_key shape_id
132   cbb99ed41d87d6c52dc44fd909580eb2     None
1663  8510daa6c8576e648fcbd4f92ea73a51     None
5688  54b4fbf52fa927769d56a3dad40b8c43     None
9615  f52fb3c8c9f650e6f2485086b075a340     None


In [13]:
route_id_shapes_geom = route_id_shapes.merge(
    shapes_unique[
        ["feed_key", "shape_id", "geometry"]
    ],
    on=["feed_key", "shape_id"],
    how="left"
)

In [14]:
route_id_shapes_geom = route_id_shapes_geom.dropna(
    subset=["geometry"]
).copy()

In [15]:
route_id_shapes_geom = gpd.GeoDataFrame(
    route_id_shapes_geom,
    geometry="geometry",
    crs="EPSG:4326"
)

In [16]:
route_shapes_plot = (
    route_id_shapes_geom[
        ["feed_key", "route_id", "shape_id", "geometry"]
    ]
    .drop_duplicates(
        subset=["feed_key", "route_id", "shape_id"]
    )
    .copy()
)

route_shapes_plot = gpd.GeoDataFrame(
    route_shapes_plot,
    geometry="geometry",
    crs="EPSG:4326"
)

In [17]:
route_shapes_plot.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 11219 entries, 0 to 215576
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   feed_key  11219 non-null  object  
 1   route_id  11219 non-null  object  
 2   shape_id  11219 non-null  object  
 3   geometry  11219 non-null  geometry
dtypes: geometry(1), object(3)
memory usage: 438.2+ KB


In [18]:
def export_gdf(gdf, filename: str):
    # Save GeoParquet locally
    gdf.to_parquet(f"{filename}.parquet", engine="pyarrow", index=False)

    # Full GCS path including folder
    gcs_path = f"{GCS_FILE_PATH}/transit_provider_dashboard/{filename}.parquet"

    # Upload to GCS
    fs.put(
        f"{filename}.parquet",
        gcs_path,
        token=credentials.token
    )

    # Remove local file
    os.remove(f"{filename}.parquet")
    print(f"saved {gcs_path}")

In [19]:
# Store data in warehouse
export_gdf(route_shapes_plot, "route_id_shapes")

saved gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/route_id_shapes.parquet


In [20]:
# Querying unique GTFS stops (lightweight)
with db_engine.connect() as connection:
    query = """
        SELECT
            name,
            stop_id,
            stop_key,
            stop_code,
            stop_name,
            location_type,
            route_id_array,
            ANY_VALUE(pt_geom) AS pt_geom
        FROM
            cal-itp-data-infra.mart_gtfs_rollup.fct_monthly_scheduled_stops
        WHERE(
        month_first_day BETWEEN DATE('2025-01-01') AND DATE('2025-12-01') 
        OR month_first_day BETWEEN DATE('2026-03-01') AND DATE('2026-06-01')
        )
        GROUP BY
            name,
            stop_id,
            stop_key,
            stop_code,
            stop_name,
            route_id_array,
            location_type
    """
    stops_unique = pd.read_sql(query, connection)



In [21]:
with db_engine.connect() as connection:
    query_bridge = """
        SELECT *
        FROM cal-itp-data-infra-staging.mart_transit_database.bridge_gtfs_analysis_name_x_ntd
    """
    bridge_gtfs = pd.read_sql(query_bridge, connection)

# save as parquet
bridge_gtfs.to_parquet("bridge_gtfs_analysis_name_x_ntd.parquet", engine="pyarrow", index=False)

In [22]:
from shared_utils import geo_utils

In [23]:
stops_unique_gdf = (
    stops_unique
    .pipe(
        geo_utils.convert_to_gdf,
        geom_col="pt_geom",
        geom_type="point"
    )
)

In [24]:
stops_unique_gdf = stops_unique_gdf.drop_duplicates(subset=["stop_id", "stop_name"])

In [25]:
stops_with_crosswalk = stops_unique_gdf.merge(
    bridge_gtfs,
    left_on="name",       
    right_on="schedule_gtfs_dataset_name",  
    how="left"             
)

In [26]:
# List of variant rows to drop (no org info, but other variants in group have org info)
drop_names = [
    'Anaheim Resort Schedule v2', 'Tri-Valley Wheels Schedule', 'Lynwood Schedule',
    'Mountain Transit GMV Schedule', 'Nevada County Remix Schedule',
    'Roseville Transit GMV Schedule', 'Roseville Transit TripShot Schedule',
    'TCRTA Schedule Historic 2', 'Tahoe Transportation District GMV Schedule',
    'Victor Valley GMV Schedule', 'eTrans Schedule Remix',
    'eTrans Schedule Trillium', 'Bellflower Remix Schedule',
    'Capitol Corridor Schedule', 'Commute.org Schedules',
    'El Monte RTAP Schedule', 'Vacaville Schedule', 'Marin GMV Schedule',
    'Petaluma GMV Schedule', 'Union City GMV Schedule',
    'South San Francisco Schedule', 'Basin Transit GMV Schedule',
    'Vine GMV Schedule', 'Caltrain Schedule',
    'Big Blue Bus Swiftly Schedule', 'Emery Go-Round TripShot Schedule',
    'Santa Rosa CityBus GMV Schedule', 'Merced GMV Schedule',
    'SolTrans Schedule', 'eTrans Schedule', 'WestCAT Schedule', 'MVGO Schedule',
    'Cerritos on Wheels Schedule', 'San Francisco Bay Ferry Schedule',
    'SMART Schedule', 'Lawndale Beat GMV Schedule',
    'Mountain View Community Shuttle Schedule', 'Desert Roadrunner GMV Schedule',
    'ACE Schedule', 'AC Transit Schedule', 'SCVTA Schedule', 'SamTrans Schedule',
    'Amtrak San Joaquins Schedule', 'Fairfield Schedule', 'Rio Vista Schedule',
    'County Connection Schedule', 'TCRTA TripShot Schedule'
]
# Drop these rows
stops_with_crosswalk_cleaned = stops_with_crosswalk[~stops_with_crosswalk['name'].isin(drop_names)].reset_index(drop=True)


In [27]:
def export_gdf(gdf, filename: str):
    # Save GeoParquet locally
    gdf.to_parquet(f"{filename}.parquet", engine="pyarrow", index=False)

    # Full GCS path including folder
    gcs_path = f"{GCS_FILE_PATH}/transit_provider_dashboard/{filename}.parquet"

    # Upload to GCS
    fs.put(
        f"{filename}.parquet",
        gcs_path,
        token=credentials.token
    )

    # Remove local file
    os.remove(f"{filename}.parquet")
    print(f"saved {gcs_path}")

In [28]:
# Store data in warehouse
export_gdf(stops_with_crosswalk_cleaned, "stop_data_cleaned_with_route_array")

saved gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/stop_data_cleaned_with_route_array.parquet


In [29]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/census_tracts_data_2024.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [30]:
# Load the stored organization dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/organization_data_2025_10_16.parquet", "rb") as f:
    valid_organization_full = pd.read_parquet(f)

In [31]:
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/ridership_data_2024.parquet", "rb") as f:
    ridership_data_grouped = pd.read_parquet(f)

In [32]:
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/stop_data_cleaned_with_route_array.parquet", "rb") as f:
    orgs_stops_clean = gpd.read_parquet(f)

In [33]:
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/census_ca_data_2024.parquet", "rb") as f:
    ca_totals_acs = gpd.read_parquet(f)

In [34]:
columns_to_keep = [
    "name", "ntd_id", "ntd_id_2022", "stop_id", "stop_name", "route_id_array",
    "schedule_gtfs_dataset_name", "organization_source_record_id", 
    "geometry", "analysis_name", "organization_name"
]

stops_clean_subset = orgs_stops_clean[columns_to_keep].copy()

In [35]:
# Clean names to remove noise words
remove_words = ['Schedule', 'GMV', 'TripShot', 'Remix', 'v2', 'Historic', 'Cal-ITP', 'RTAP']

def clean_name(name):
    for w in remove_words:
        name = re.sub(rf'\b{w}\b', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

In [36]:
import re
from rapidfuzz import process, fuzz

In [37]:
# Add cleaned columns
stops_clean_subset['name_clean'] = stops_clean_subset['name'].apply(clean_name)
valid_organization_full['name_clean'] = valid_organization_full['name'].apply(clean_name)

In [38]:
base = stops_clean_subset.copy()
valid_names_clean = valid_organization_full["name_clean"].tolist()



def fuzzy_match_org(name, valid_names, threshold=90):
    match = process.extractOne(name, valid_names, scorer=fuzz.WRatio)

    if match and match[1] >= threshold:
        matched_name_clean = match[0]

        idx = valid_organization_full.index[
            valid_organization_full["name_clean"] == matched_name_clean
        ][0]

        return (
            valid_organization_full.loc[idx, "source_record_id"],
            valid_organization_full.loc[idx, "name"]
        )

    return None, None



target_names = [
    "Auburn Schedule",
    "Fairfield Schedule",
    "Gold Coast Schedule",
    "Golden Gate Bridge Schedule",
    "Moorpark Schedule",
    "Morro Bay Cal-ITP Schedule",
    "Simi Valley Schedule",
    "South San Francisco Schedule",
    "Trinity Schedule",
    "Thousand Oaks Schedule"
]

mask = (
    base["organization_source_record_id"].isna()
    & base["name"].isin(target_names)
)


In [39]:
subset = base.loc[mask].copy()

matches = subset["name_clean"].apply(
    lambda x: fuzzy_match_org(x, valid_names_clean)
)

subset["organization_source_record_id"] = [m[0] for m in matches]
subset["analysis_name"] = [m[1] for m in matches]

In [40]:
base.loc[mask, "organization_source_record_id"] = subset["organization_source_record_id"]
base.loc[mask, "analysis_name"] = subset["analysis_name"]

In [41]:
orgs_stops_merged = base.merge(
    valid_organization_full[
        ["source_record_id", "key", "organization_type", "ntd_id", "ntd_id_2022"]
    ],
    left_on="organization_source_record_id",
    right_on="source_record_id",
    how="left"
)

In [42]:
filled_rows = subset[subset["organization_source_record_id"].notna()]

filled_unique = (
    filled_rows[
        ["name", "organization_name", "organization_source_record_id", "analysis_name"]
    ]
    .drop_duplicates()
    .sort_values("name")
)

print(f"Filled rows: {filled_unique['name'].nunique()}")
filled_unique

Filled rows: 7


,name,organization_name,organization_source_record_id,analysis_name
14141,Auburn Schedule,None,recbW86Xrtuw8PhiU,City of Auburn
5,Gold Coast Schedule,None,recS7GnKTcQVX20HE,Gold Coast Transit District
0,Golden Gate Bridge Schedule,None,recoX7qMhlPrgfuz3,"Golden Gate Bridge, Highway and Transportation..."
246,Moorpark Schedule,None,recojKzQsBzE1hjVu,City of Moorpark
2096,Morro Bay Cal-ITP Schedule,None,recH53ghrYpk4gKhe,City of Morro Bay
21,Simi Valley Schedule,None,rec1ErIn9gG1Isk5W,City of Simi Valley
51,Thousand Oaks Schedule,None,recPJULRJk1Yn824N,City of Thousand Oaks


In [43]:
# Find rows where organization_source_record_id is still missing
still_missing = orgs_stops_merged[orgs_stops_merged['organization_source_record_id'].isna()]

# Get unique names
unique_missing_names = still_missing['name'].unique()

print(f"Total unique names still missing: {len(unique_missing_names)}")
print(unique_missing_names)

Total unique names still missing: 34
['Dana Point Trolley PassioGo Schedule' 'SLO Schedule'
 'Anaheim Resort Schedule' 'Dana Point Trolley Schedule' 'Vine Schedule'
 'Clovis PassioGo Schedule' 'Sonoma County Transit Schedule'
 'Marin Optibus Schedule' 'Yuba-Sutter PassioGo Schedule'
 'LAX Shuttles Schedule' 'Cerritos on Wheels Website Schedule'
 'VCTC Schedule' 'Rosemead Passio Schedule' 'Trinity Schedule'
 'Nevada County Schedule' 'Peppedine University Shuttles Schedule'
 'Fric and Frac Schedule' 'BART Schedule' 'Trinity Remix Schedule'
 'University of San Diego Tram Services Schedule'
 'Union City TripShot Schedule' 'CSULB Shuttle PassioGo Schedule'
 'Bay Area 511 Emery Express Schedule' 'LAX FlyAway Schedule'
 'Bellflower Bus Schedule' 'Beaumont Pass Schedule'
 'Guadalupe Flyer Schedule' 'Laguna Niguel Trolley PassioGo Schedule'
 'Bulldog Express PassioGo Schedule' 'Laguna Niguel Schedule'
 'Bay Area 511 Regional Schedule' 'Sonoma Schedule' 'VCTC GMV Schedule'
 'Ridgecrest Schedule'

In [44]:
# Manual mapping for remaining unmatched
manual_matches = {
    # "BART Schedule": "San Francisco Bay Area Rapid Transit District",
    # "Vine Schedule": "Napa Valley Transportation Authority",
    # "VCTC Schedule": "Ventura County Transportation Commission",
    "Sonoma County Transit Schedule": "Sonoma County"
    
}

# Loop over manual matches and update orgs_stops_merged
for gtfs_name, org_name in manual_matches.items():
    # Get valid_organization_full row
    org_row = valid_organization_full[valid_organization_full['name'] == org_name]
    
    if not org_row.empty:
        source_id = org_row['source_record_id'].values[0]
        key = org_row['key'].values[0]
        org_type = org_row['organization_type'].values[0]
        
        # Update orgs_stops_merged where the GTFS name matches
        mask = orgs_stops_merged['name'] == gtfs_name
        orgs_stops_merged.loc[mask, 'organization_source_record_id'] = source_id
        orgs_stops_merged.loc[mask, 'analysis_name'] = org_name
        orgs_stops_merged.loc[mask, 'key'] = key
        orgs_stops_merged.loc[mask, 'organization_type'] = org_type



In [45]:
orgs_stops_merged.name.nunique()

232

In [46]:
orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Anaheim Resort Schedule",
    "analysis_name"
] = "Anaheim Transportation Network"


orgs_stops_merged.loc[
    orgs_stops_merged["name"].isin(["SLO Schedule", "SLO Peak Transit Schedule"]),
    "analysis_name"
] = "City of San Luis Obispo"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Bay Area 511 Dumbarton Express Schedule",
    "analysis_name"
] = "Dumbarton Bridge Regional Operations Consortium"


orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Bay Area 511 Sonoma County Transit Schedule",
    "analysis_name"
] = "Cloverdale Transit"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "South County Transit Link Schedule",
    "analysis_name"
] = "South County Transit Link"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Bay Area 511 Golden Gate Transit Schedule",
    "analysis_name"
] = "Golden Gate Transit"


orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Golden Gate Bridge Schedule",
    "analysis_name"
] = "Golden Gate Bridge"


orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Bay Area 511 Golden Gate Ferry Schedule",
    "analysis_name"
] = "Golden Gate Ferry"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "LA Metro Bus Schedule",
    "organization_name"
] = "LA Metro Bus"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "LA Metro Rail Schedule",
    "organization_name"
] = "LA Metro Rail"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Golden Gate Park Shuttle Schedule",
    "analysis_name"
] = "SF Golden Gate Park Shuttle"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "Bay Area 511 Muni Schedule",
    "analysis_name"
] = "SF Muni"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "SLO Peak Transit Schedule",
    "analysis_name"
] = "SLO Peak Transit"

orgs_stops_merged.loc[
    orgs_stops_merged["name"] == "SLO Schedule",
    "analysis_name"
] = "City of San Luis Obispo"


In [47]:
# Reproject to match census tracts CRS
orgs_stops_merged = orgs_stops_merged.to_crs(tracts_ca_acs.crs)

In [48]:
orgs_stop_buffered = orgs_stops_merged.copy()
orgs_stop_buffered["geometry"] = orgs_stop_buffered.geometry.buffer(8046.72)

In [49]:
orgs_stop_dissolved = orgs_stop_buffered.dissolve(by='name')

In [50]:
orgs_stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 114146 entries, 0 to 114145
Data columns (total 17 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   name                           114146 non-null  object  
 1   ntd_id_x                       84438 non-null   object  
 2   ntd_id_2022_x                  84688 non-null   object  
 3   stop_id                        114146 non-null  object  
 4   stop_name                      114146 non-null  object  
 5   route_id_array                 114146 non-null  object  
 6   schedule_gtfs_dataset_name     95748 non-null   object  
 7   organization_source_record_id  96894 non-null   object  
 8   geometry                       114146 non-null  geometry
 9   analysis_name                  97138 non-null   object  
 10  organization_name              95748 non-null   object  
 11  name_clean                     114146 non-null  object  
 12  source_r

In [51]:
orgs_stop_dissolved = orgs_stop_dissolved.reset_index()

In [52]:
# Compute the intersection between buffered stops and census tracts.
geometry_intersect = gpd.overlay(
    orgs_stop_dissolved, 
    tracts_ca_acs, 
    how = 'intersection', 
    keep_geom_type=True)

In [53]:
# Calculate the area of each intersected geometry in square meters.
geometry_intersect['area_2'] = geometry_intersect.geometry.area

In [54]:
# Adjust total population by the proportion of the tract area that intersects the stop buffer.
# Calculate the proportion of each tract's area that intersects the stop buffer
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']

In [55]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'poverty_pop', 'non_us_citizen', 'workers_with_no_car', 
    'households_with_no_cars', 'disabled_pop', 'public_asst_pop', 
    'inc_extremelylow', 'inc_verylow', 'inc_low', 
    'male_seniors', 'female_seniors', 'veteran_pop', 'male_youth',  'female_youth', 'veteran_above_65',
]

# Apply area ratio to create adjusted metrics
geometry_intersect[[f'{col}_adj' for col in cols_to_weight]] = (
    geometry_intersect[cols_to_weight].multiply(geometry_intersect['area_ratio'], axis=0)
)

In [56]:
# Stop level demography data 
filtered_final_data = geometry_intersect[['name', 'organization_type', 'organization_name',  'analysis_name', 'ntd_id_y', 'ntd_id_2022_y', 
                                          'stop_id', 'stop_name', "route_id_array",
                                         'GEOIDFQ', 'geometry', 'area_2',	'total_pop_adj',	'poverty_pop_adj',	
                                          'non_us_citizen_adj',	'workers_with_no_car_adj',	'households_with_no_cars_adj',	'disabled_pop_adj',	
                                          'public_asst_pop_adj', 'inc_extremelylow_adj', 'inc_verylow_adj',	'inc_low_adj',	'male_seniors_adj',	
                                          'female_seniors_adj', 'male_youth_adj',  'female_youth_adj', 'veteran_pop_adj', 'veteran_above_65_adj' ]]

filtered_final_data.head(2)

,name,organization_type,organization_name,analysis_name,ntd_id_y,ntd_id_2022_y,stop_id,stop_name,route_id_array,GEOIDFQ,geometry,area_2,total_pop_adj,poverty_pop_adj,non_us_citizen_adj,workers_with_no_car_adj,households_with_no_cars_adj,disabled_pop_adj,public_asst_pop_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,male_seniors_adj,female_seniors_adj,male_youth_adj,female_youth_adj,veteran_pop_adj,veteran_above_65_adj
0,Alhambra Schedule,None,City of Alhambra,City of Alhambra,None,None,2619847,Valley & Edgewood,[GreenLine],1400000US06037481712,"POLYGON ((173817.149 -436699.032, 174163.881 -...",723082.167545,5269.0,1053.0,1612.0,91.0,356.0,780.0,403.0,1932.0,971.0,415.0,505.0,513.0,348.0,439.0,69.0,20.0
1,Alhambra Schedule,None,City of Alhambra,City of Alhambra,None,None,2619847,Valley & Edgewood,[GreenLine],1400000US06037204920,"POLYGON ((165983.786 -442389.657, 166019.401 -...",907841.292393,2274.0,329.0,444.0,33.0,107.0,361.0,203.0,689.0,511.0,200.0,148.0,292.0,178.0,151.0,34.0,26.0


In [57]:
group_key = ['name']

# Identify adjusted demographic columns
adj_cols = [col for col in geometry_intersect.columns if col.endswith('_adj')]

# Extra non-demographic attributes to keep (take first occurrence per agency)
extra_cols = ['organization_type', 'ntd_id_y', 'ntd_id_2022_y', 'key']

# Dissolve stop buffers to get agency shapes
agency_geometry = orgs_stop_dissolved.dissolve(by=group_key, as_index=False)

# --- DROP overlapping extra columns from agency_geometry ---
agency_geometry = agency_geometry.drop(columns=extra_cols, errors='ignore')

# Aggregate population metrics
agency_demo = geometry_intersect.groupby(group_key, as_index=False)[adj_cols].sum()

# Merge demographics with geometry
agency_summary = agency_geometry.merge(agency_demo, on=group_key, how='left')

# Merge extra attributes (take first)
extra_attrs = orgs_stop_dissolved[group_key + extra_cols].drop_duplicates(subset=group_key)
agency_summary = agency_summary.merge(extra_attrs, on=group_key, how='left')




In [58]:
agency_summary = gpd.GeoDataFrame(
    agency_summary,
    geometry='geometry',
    crs=tracts_ca_acs.crs
).to_crs(epsg=4326)



In [59]:
agency_summary = agency_summary.drop(columns=["ntd_id_x", "ntd_id_2022_x"])

agency_summary = agency_summary.rename(
    columns={
        "ntd_id_y": "ntd_id",
        "ntd_id_2022_y": "ntd_id_2022"
    }
)


In [60]:
# Merge acs and ntd data 
merged_agency_ntd = (
    pd.merge(
        agency_summary,
        ridership_data_grouped,
        how='left',
        left_on='ntd_id_2022',
        right_on='ntd_id'
    )
    .sort_values(by='agency')
)

In [61]:
merged_agency_ntd = gpd.GeoDataFrame(
    merged_agency_ntd, 
    geometry='geometry', 
    crs=agency_summary.crs
)

In [62]:
agency_map_ntd = {
    "Plumas Transit Systems": {"ntd_id_2022": "91119", "upt": 29264, "voms": 6},
    "Redwood Coast Transit Authority": {"ntd_id_2022": "91097", "upt": 64681, "voms": 4},
    "Nevada County": {"ntd_id_2022": "91095", "upt": 146562, "voms": 17},
    "Lake Transit Authority": {"ntd_id_2022": "91053", "upt": 228536, "voms": 26},
    "Mendocino Transit Authority": {"ntd_id_2022": "91047", "upt": 153720, "voms": 20},
    "San Benito County Local Transportation Authority": {"ntd_id_2022": "91009", "upt": 81284, "voms": 19},
    "Tulare County Regional Transit Agency": {"ntd_id_2022": "90310", "upt": 677529, "voms": 45},
    "City of Baldwin Park": {"ntd_id_2022": "90251", "upt": 78209, "voms": 8},
    "Marin County Transit District": {"ntd_id_2022": "90234", "upt": 2867835, "voms": 72},
    "Anaheim Transportation Network": {"ntd_id_2022": "90211", "upt": 8696858, "voms": 62},
    "Eastern Contra Costa Transit Authority": {"ntd_id_2022": "90162", "upt": 1411611, "voms": 97},
    "Redding Area Bus Authority": {"ntd_id_2022": "90093", "upt": 503018, "voms": 23},
    "Sonoma County": {"ntd_id_2022": "90089", "upt": 787144, "voms": 56},
    "Yuba-Sutter Transit Authority": {"ntd_id_2022": "90061", "upt": 573966, "voms": 29},
    "San Diego Metropolitan Transit System, Airport, Flagship Cruises": {"ntd_id_2022": "90026", "upt": 75682794, "voms": 819},
    "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)": {"ntd_id_2022": "90164", "upt": 454893, "voms": 42},    
    "Sacramento Regional Transit District": {"ntd_id_2022": "90019", "upt": 16361582, "voms": 387}
}

In [63]:
for agency, vals in agency_map_ntd.items():
    mask = merged_agency_ntd["analysis_name"].eq(agency)

    merged_agency_ntd.loc[mask, "ntd_id_2022"] = vals["ntd_id_2022"]
    merged_agency_ntd.loc[mask, "unlinked_passenger_trips_upt"] = vals["upt"]
    merged_agency_ntd.loc[mask, "agency_voms"] = vals["voms"]

In [64]:
updated = merged_agency_ntd[
    merged_agency_ntd["analysis_name"].isin(agency_map_ntd.keys())
]

In [65]:
merged_agency_ntd = merged_agency_ntd[['key', 'name', 'organization_type', 'organization_name', 'analysis_name', 'ntd_id_2022', 'agency', 
                                       'total_pop_adj', 'poverty_pop_adj', 'non_us_citizen_adj', 'workers_with_no_car_adj',
        'households_with_no_cars_adj', 'disabled_pop_adj',
       'public_asst_pop_adj', 'inc_extremelylow_adj', 'inc_verylow_adj',
       'inc_low_adj', 'male_seniors_adj', 'female_seniors_adj', 'male_youth_adj', 'female_youth_adj',
       'veteran_pop_adj', 'veteran_above_65_adj', 'unlinked_passenger_trips_upt', 'agency_voms']]

In [66]:
#Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/merged_agency_ntd_route_array.parquet", "wb") as f:
    merged_agency_ntd.to_parquet(f, index=False)

In [67]:
def export_gdf(gdf, filename: str, export_csv: bool = True):
    # Update the path
    gcs_target_path = f"{GCS_FILE_PATH}/transit_provider_dashboard/"

    # Export as Parquet
    parquet_file = f"{filename}.parquet"
    gdf.to_parquet(parquet_file, engine="pyarrow", index=False)
    
    fs.put(
        parquet_file,
        f"{gcs_target_path}{parquet_file}",
        token=credentials.token
    )
    os.remove(parquet_file)
    print(f"Saved Parquet: {gcs_target_path}{parquet_file}")
    
    if export_csv:
        # Export as CSV
        csv_file = f"{filename}.csv"
        gdf.to_csv(csv_file, index=False)
        
        fs.put(
            csv_file,
            f"{gcs_target_path}{csv_file}",
            token=credentials.token
        )
        os.remove(csv_file)
        print(f"Saved CSV: {gcs_target_path}{csv_file}")




In [68]:
# Store data in warehouse
export_gdf(merged_agency_ntd, "merged_agency_ntd_route_array")

Saved Parquet: gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/merged_agency_ntd_route_array.parquet
Saved CSV: gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/merged_agency_ntd_route_array.csv


In [69]:
# Store data in warehouse
export_gdf(orgs_stop_buffered, "organization_stops_buffered_route_array")

Saved Parquet: gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/organization_stops_buffered_route_array.parquet
Saved CSV: gs://calitp-analytics-data/data-analyses/transit_provider_dashboard/organization_stops_buffered_route_array.csv
